In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 3
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
18.820810607933822

Trial 1 =========================================
17.574536315869977

Trial 2 =========================================
14.628480256057177

Trial 3 =========================================
13.202116379372134

Trial 4 =========================================
14.246837555745126

Trial 5 =========================================
15.399026819411183

Trial 6 =========================================
15.010153740575795

Trial 7 =========================================
16.055000561433864

Trial 8 =========================================
14.975339108927136

Trial 9 =========================================
18.659751007445223

Trial 10 =========================================
16.255847034664367

Trial 11 =========================================
14.4386844307931

Trial 12 =========================================
14.287765227963185

Trial 13 =========================================
16.237226807559544

Trial 14 =========

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/mi

Trial 32 =========================================
16.24307998649087



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 33 =========================================
15.372099196368787

Trial 34 =========================================
14.455572310043646

Trial 35 =========================================
17.56550433059252

Trial 36 =========================================
14.709501267448463

Trial 37 =========================================
16.182149472906247

Trial 38 =========================================
15.38197197170858

Trial 39 =========================================
15.009786736377652



...

Trial 82 =========================================
16.22195899620138

Trial 83 =========================================
15.376620396163455

Trial 84 =========================================
16.189811080012078

Trial 85 =========================================
15.340890507496821

Trial 86 =========================================
17.60126286991622



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 87 =========================================
15.57238125744904

Trial 88 =========================================
15.16292338950508

Trial 89 =========================================
18.769756765529184

Trial 90 =========================================
18.809850580622435

Trial 91 =========================================
14.80040065312573

Trial 92 =========================================
14.241969864138843

Trial 93 =========================================
18.8135883163985

Trial 94 =========================================
18.811486210032083

Trial 95 =========================================
15.139956893094642

Trial 96 =========================================
14.770062879810114

Trial 97 =========================================
14.830014200676597

Trial 98 =========================================
14.445811534638517



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 99 =========================================
16.248858488264904

[18.82081061 17.57453632 14.62848026 13.20211638 14.24683756 15.39902682
 15.01015374 16.05500056 14.97533911 18.65975101 16.25584703 14.43868443
 14.28776523 16.23722681 15.3884378  13.18507986 16.01457663 14.62979751
 14.71326107 15.39835345 15.39731811 16.22662331 16.24341268 17.51620329
 15.36148283 18.79973128 13.90876439 15.57388991 14.24024763 13.42691633
 13.38134971 17.50312343 16.24307999 15.3720992  14.45557231 17.56550433
 14.70950127 16.18214947 15.38197197 15.00978674 16.20649238 15.38670359
 14.50059707 15.38700676 16.21623385 14.44611286 18.72796784 17.07252228
 15.21290729 14.25463647 13.5405725  15.29864938 15.9205202  14.44425453
 16.22611703 15.39749566 14.85419425 13.14270445 14.44611397 13.43956311
 15.22252202 15.92922201 13.14661335 14.46517952 18.7961233  15.91053801
 13.87926183 13.4335544  15.35335355 16.0527744  13.42311987 15.92922201
 14.26761684 14.29783894 18.75541736 14.44530705 15.9

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.820810607933822
Avg = 15.523718508889282
Std = 1.5100802908349074


In [6]:
print(y_max_arr.tolist())

[18.820810607933822, 17.574536315869977, 14.628480256057177, 13.202116379372134, 14.246837555745126, 15.399026819411183, 15.010153740575795, 16.055000561433864, 14.975339108927136, 18.659751007445223, 16.255847034664367, 14.4386844307931, 14.287765227963185, 16.237226807559544, 15.38843779804611, 13.18507986414777, 16.01457662919788, 14.629797514110646, 14.713261067104181, 15.398353452586232, 15.39731811180035, 16.226623313090744, 16.24341267778319, 17.51620329260204, 15.361482825194521, 18.799731275979877, 13.90876439136002, 15.573889913867648, 14.240247627398562, 13.426916328400495, 13.381349707459359, 17.503123433469728, 16.24307998649087, 15.372099196368787, 14.455572310043646, 17.56550433059252, 14.709501267448463, 16.182149472906247, 15.38197197170858, 15.009786736377652, 16.206492383143296, 15.386703585539719, 14.5005970686232, 15.387006755588926, 16.216233847499907, 14.446112856873697, 18.72796784335983, 17.07252228031689, 15.212907293339045, 14.254636465207776, 13.540572496638

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)